# Quality Research

## Stage 1C — frozen PIT Profitability

This notebook implements only the raw component `NI_TTM_PIT / AverageAssets_PIT`. It does not implement Cash Realization, Balance-sheet Resilience, a Quality composite, ranking, normalization, `X_char`, weights, or a backtest.


## 1. Information set and raw evidence

The decision is at an XNYS regular-session open. An Assets reference date is an economic balance-sheet instant; it is distinct from filing vintage and PIT eligibility. Only the first eligible XNYS open strictly after SEC acceptance is eligible.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd

FORMS = {'10-Q', '10-K', '10-Q/A', '10-K/A'}
RAW_ROOT = next(p / 'data' / 'raw' / 'stage_0b_sec' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw' / 'stage_0b_sec' / 'manifest.json').exists())
PROJECT_ROOT = RAW_ROOT.parents[2]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from pit_profitability import profitability_as_of
from xnys_pit import eligibility_from_acceptance

manifest = json.loads((RAW_ROOT / 'manifest.json').read_text(encoding='utf-8'))
def read_raw(entry):
    return json.loads((PROJECT_ROOT / entry['local_raw_file']).read_text(encoding='utf-8'))

resolved = {x['issuer']: x['resolved_cik'] for x in manifest if x['endpoint'] == 'identity_resolution'}
payloads, filing_rows, seen = {}, [], set()
for item in manifest:
    if item['request_status'] != 'HTTP_200' or not item.get('local_raw_file'):
        continue
    if item['endpoint'] == 'companyfacts':
        payload = read_raw(item)
        if str(payload.get('cik', '')).zfill(10) == resolved.get(item['issuer'], str(payload.get('cik', '')).zfill(10)):
            payloads[item['issuer']] = payload
    if item['endpoint'] in {'submissions', 'submissions_history'} and item['content_sha256'] not in seen:
        seen.add(item['content_sha256'])
        inventory = read_raw(item).get('filings', {}).get('recent', read_raw(item))
        for values in zip(*inventory.values()):
            row = dict(zip(inventory, values))
            if row.get('form') in FORMS:
                filing_rows.append({'issuer': item['issuer'], **row})

filings = pd.DataFrame(filing_rows).drop_duplicates(['issuer', 'accessionNumber'])[['issuer', 'accessionNumber', 'form', 'filingDate', 'acceptanceDateTime']]
filings['acceptanceDateTime'] = pd.to_datetime(filings['acceptanceDateTime'], utc=True)
print(f'Reading immutable SEC evidence from {RAW_ROOT.relative_to(PROJECT_ROOT)}')


In [ ]:
asset_rows, income_rows = [], []
for issuer, payload in payloads.items():
    cik = str(payload['cik']).zfill(10)
    for unit, facts in payload.get('facts', {}).get('us-gaap', {}).get('Assets', {}).get('units', {}).items():
        for fact in facts:
            if fact.get('form') in FORMS:
                asset_rows.append({'issuer': issuer, 'cik': cik, 'tag': 'Assets', 'unit': unit, 'reference_date': fact.get('end'), 'value': fact.get('val'), 'form': fact.get('form'), 'accn': fact.get('accn')})
    for unit, facts in payload.get('facts', {}).get('us-gaap', {}).get('NetIncomeLoss', {}).get('units', {}).items():
        for fact in facts:
            if fact.get('form') in FORMS:
                income_rows.append({'issuer': issuer, 'cik': cik, 'tag': 'NetIncomeLoss', 'unit': unit, 'start': fact.get('start'), 'end': fact.get('end'), 'value': fact.get('val'), 'form': fact.get('form'), 'accn': fact.get('accn')})

lookup = filings.rename(columns={'accessionNumber': 'accn', 'form': 'filing_form'})
assets = pd.DataFrame(asset_rows).merge(lookup, on=['issuer', 'accn'], how='left', validate='many_to_one')
net_income = pd.DataFrame(income_rows).merge(lookup, on=['issuer', 'accn'], how='left', validate='many_to_one')
assets['PIT eligibility'] = pd.to_datetime(assets['acceptanceDateTime'], utc=True).map(lambda accepted: eligibility_from_acceptance(accepted).portfolio_decision_time)
assets.groupby('issuer').agg(assets_facts=('accn', 'size'), reference_dates=('reference_date', 'nunique'), amendments=('form', lambda x: x.str.endswith('/A').sum()))


## 2. AAPL transparent calculation

At AAPL's 2025-11-03 XNYS open, the function validates the PIT TTM and requires Assets exactly at the calendar day before its economic start and at its economic end. It does not use the merely latest two Assets observations.


In [ ]:
AAPL_DECISION = pd.Timestamp('2025-11-03T14:30:00Z')
aapl = profitability_as_of(net_income, assets, 'AAPL', AAPL_DECISION)
assert aapl.status == 'VALID'

manual_average_assets = (aapl.average_assets.beginning_assets.value + aapl.average_assets.ending_assets.value) / 2
manual_profitability = aapl.net_income.value / manual_average_assets
np.testing.assert_allclose(manual_average_assets, aapl.average_assets.average_assets)
np.testing.assert_allclose(manual_profitability, aapl.profitability)

aapl_trace = pd.DataFrame([
    {'component': 'NI_TTM_PIT', 'value': aapl.net_income.value, 'economic_period': f'{aapl.average_assets.ttm_economic_start.date()} to {aapl.average_assets.ttm_economic_end.date()}', 'accession': aapl.net_income.quarters.iloc[-1].source_accessions, 'PIT_eligibility': aapl.net_income.quarters.iloc[-1].pit_eligibility},
    {'component': 'Assets_begin_PIT', 'value': aapl.average_assets.beginning_assets.value, 'economic_period': aapl.average_assets.beginning_assets.reference_date.date(), 'accession': aapl.average_assets.beginning_assets.source_accession, 'PIT_eligibility': aapl.average_assets.beginning_assets.pit_eligibility},
    {'component': 'Assets_end_PIT', 'value': aapl.average_assets.ending_assets.value, 'economic_period': aapl.average_assets.ending_assets.reference_date.date(), 'accession': aapl.average_assets.ending_assets.source_accession, 'PIT_eligibility': aapl.average_assets.ending_assets.pit_eligibility},
    {'component': 'AverageAssets_PIT (manual = implementation)', 'value': manual_average_assets, 'economic_period': None, 'accession': None, 'PIT_eligibility': None},
    {'component': 'Profitability (manual = implementation)', 'value': manual_profitability, 'economic_period': None, 'accession': None, 'PIT_eligibility': None},
])
aapl_trace


The two deterministic assertions are the notebook-to-implementation consistency check: `AverageAssets_PIT = (Assets_begin + Assets_end) / 2` and `Profitability = NI_TTM_PIT / AverageAssets_PIT`.


## 3. Cross-issuer implementation validation

This validates provenance and construction only. It does not rank issuers or make a future-return claim.


In [ ]:
decision_dates = {'AAPL': '2025-11-03T14:30:00Z', 'JPM': '2026-02-17T14:30:00Z', 'XOM': '2026-02-19T14:30:00Z', 'UNH': '2026-03-02T14:30:00Z', 'WMT': '2026-03-16T13:30:00Z'}
rows = []
for issuer, decision in decision_dates.items():
    result = profitability_as_of(net_income, assets, issuer, decision)
    rows.append({'issuer': issuer, 'decision_date': result.decision_time, 'NI_TTM_PIT': result.net_income.value, 'Assets_begin': result.average_assets.beginning_assets.value, 'Assets_end': result.average_assets.ending_assets.value, 'AverageAssets_PIT': result.average_assets.average_assets, 'Profitability': result.profitability, 'status': result.status})
cross_issuer_validation = pd.DataFrame(rows)
cross_issuer_validation


## 4. Explicit NA policy

The five selected recent traces have exact valid endpoints, so the preserved representative sample has no naturally occurring invalid denominator in this narrow check. The reusable tests explicitly demonstrate mandatory `NA` behavior for a missing beginning/end boundary, a future-only comparative fact, a conflicting selected vintage, and an incompatible date. No nearest date, imputation, or retrospective rewrite is permitted.

A two-point average remains an approximation: acquisitions, disposals, consolidation-perimeter changes, large intra-period asset changes, and financial-sector balance-sheet interpretation are limitations, not permission for silent adjustments.


## 5. Frozen conclusion

**Quality / Profitability is frozen only as the raw component** `NI_TTM_PIT / AverageAssets_PIT`. Cash Realization, Balance-sheet Resilience, any Quality composite, cross-sectional transformation, and `X_char` remain unimplemented.


---

# Stage 1C audit — Operating Cash Flow for Cash Realization

Cash Realization remains a candidate, not a frozen characteristic. The economic question is: given **positive** accounting earnings, to what extent are they supported by operating cash generation? Under the frozen conceptual boundary, if `NI_TTM_PIT <= 0`, a future Cash Realization measure must be `NA`; this audit does not reinterpret negative-earnings observations through the quotient's sign.


## 6. SEC OCF tag, structure, and PIT provenance

The audited tag is `us-gaap:NetCashProvidedByUsedInOperatingActivities` for AAPL, JPM, XOM, UNH, and WMT. The raw facts retain issuer/CIK, unit, start/end, FY/FP, form, accession, acceptance datetime, and XNYS eligibility. The duration profile is inspected rather than assumed to match Net Income.


In [ ]:
from pit_net_income import ttm_net_income_as_of

OCF_TAG = 'NetCashProvidedByUsedInOperatingActivities'
ocf_rows = []
for issuer, payload in payloads.items():
    cik = str(payload['cik']).zfill(10)
    for unit, facts in payload.get('facts', {}).get('us-gaap', {}).get(OCF_TAG, {}).get('units', {}).items():
        for fact in facts:
            if fact.get('form') in FORMS:
                ocf_rows.append({'issuer': issuer, 'cik': cik, 'taxonomy_tag': OCF_TAG, 'unit': unit, 'start': fact.get('start'), 'end': fact.get('end'), 'value': fact.get('val'), 'fy': fact.get('fy'), 'fp': fact.get('fp'), 'form': fact.get('form'), 'accn': fact.get('accn'), 'frame': fact.get('frame')})

ocf_raw = pd.DataFrame(ocf_rows).merge(lookup, on=['issuer', 'accn'], how='left', validate='many_to_one')
ocf_raw['start'] = pd.to_datetime(ocf_raw['start'])
ocf_raw['end'] = pd.to_datetime(ocf_raw['end'])
ocf_raw['acceptanceDateTime'] = pd.to_datetime(ocf_raw['acceptanceDateTime'], utc=True)
ocf_raw['PIT_eligibility'] = ocf_raw['acceptanceDateTime'].map(lambda accepted: eligibility_from_acceptance(accepted).portfolio_decision_time)
ocf_raw['duration_days'] = (ocf_raw['end'] - ocf_raw['start']).dt.days
ocf_raw['duplicate_context'] = ocf_raw.duplicated(['issuer', 'unit', 'start', 'end'], keep=False)
ocf_raw.groupby(['issuer', 'form']).agg(facts=('accn', 'size'), min_duration=('duration_days', 'min'), max_duration=('duration_days', 'max'), duplicate_rows=('duplicate_context', 'sum'), amendments=('form', lambda x: x.str.endswith('/A').sum()))


Observed duration ranges are approximately 88–90 days (Q1), 180–181 days (H1 YTD), 272–273 days (9M YTD), and 363–370 days (FY), subject to issuer fiscal calendars. Thus the preserved OCF evidence ordinarily offers Q1 directly and cumulative OCF for later quarters; it does not justify assuming standalone Q2/Q3 OCF facts.


In [ ]:
# Audit-only reuse of the already validated PIT quarter mechanics: relabel the inspected OCF tag
# for the generic duration/vintage construction. This is not a Cash Realization implementation.
ocf_for_quarter_audit = ocf_raw.rename(columns={'taxonomy_tag': 'original_tag'}).assign(tag='NetIncomeLoss')

AAPL_OCF_DECISION = pd.Timestamp('2025-11-03T14:30:00Z')
aapl_ocf = ttm_net_income_as_of(ocf_for_quarter_audit, 'AAPL', AAPL_OCF_DECISION)
assert aapl_ocf.status == 'VALID'
assert aapl_ocf.quarters['economic_quarter_start'].min() == aapl.average_assets.ttm_economic_start
assert aapl_ocf.quarters['economic_quarter_end'].max() == aapl.average_assets.ttm_economic_end

aapl_ocf_quarters = aapl_ocf.quarters[['economic_quarter_start', 'economic_quarter_end', 'value', 'method', 'source_accessions', 'source_forms', 'source_acceptance_datetimes', 'pit_eligibility', 'issue']]
aapl_raw_ocf_to_ni = aapl_ocf.value / aapl.net_income.value if aapl.net_income.value > 0 else None
aapl_ocf_quarters


## 7. AAPL PIT TTM trace

At the 2025-11-03 open, AAPL has four PIT-eligible OCF quarters over the same 2024-09-29 to 2025-09-27 economic interval as the valid Net Income TTM. The audit shows Q1 direct and Q2/Q3/Q4 reconstructed only from compatible H1−Q1, 9M−H1, and FY−9M cumulative facts. The table preserves source accessions, forms, acceptance datetimes, and eligibility.


In [ ]:
aapl_ocf_summary = pd.DataFrame([{
    'decision_time': AAPL_OCF_DECISION,
    'OCF_TTM_PIT': aapl_ocf.value,
    'NI_TTM_PIT': aapl.net_income.value,
    'raw_OCF_to_NI': aapl_raw_ocf_to_ni,
    'OCF_status': aapl_ocf.status,
    'NI_status': aapl.net_income.status,
}])
aapl_ocf_summary


## 8. Cross-issuer descriptive ratio behavior

The following uses the established representative decision dates. It is descriptive only: there is no ranking, transform selection, or future-return analysis. A ratio is reported only where both TTM series are valid and Net Income is positive.


In [ ]:
ocf_ratio_rows = []
for issuer, decision in decision_dates.items():
    ni = ttm_net_income_as_of(net_income, issuer, decision)
    ocf = ttm_net_income_as_of(ocf_for_quarter_audit, issuer, decision)
    ratio = ocf.value / ni.value if ni.status == 'VALID' and ocf.status == 'VALID' and ni.value > 0 else None
    status = 'VALID' if ratio is not None else ('NI_NONPOSITIVE' if ni.status == 'VALID' and ni.value <= 0 else f'NI={ni.status}; OCF={ocf.status}')
    ocf_ratio_rows.append({'issuer': issuer, 'decision_date': pd.Timestamp(decision), 'OCF_TTM_PIT': ocf.value, 'NI_TTM_PIT': ni.value, 'raw_OCF_to_NI': ratio, 'status': status})
ocf_ratio_summary = pd.DataFrame(ocf_ratio_rows)
ocf_ratio_summary


The representative sample contains the requested structural cases: JPM has `OCF < 0` while Net Income is positive (ratio about -2.59); AAPL is approximately one (about 0.995); AAPL is also the `0 < R < 1` case; XOM, UNH, and WMT are above one (about 1.80, 1.63, and 1.90). These values show that a raw quotient is unbounded above and may be negative even under positive Net Income.

A historical descriptive scan of eligible observations from 2023 onward found observed raw ranges: AAPL 0.995–1.262, JPM -2.629–1.053, XOM 1.442–1.885, UNH 0.681–1.922, and WMT 1.789–2.303. This is an audit of structural behavior, not a cross-sectional score.


In [ ]:
# Inspect two representative unusual cases using facts in their own reported accessions.
WMT_EXTREME_DECISION = pd.Timestamp('2024-03-18T13:30:00Z')
wmt_ni = ttm_net_income_as_of(net_income, 'WMT', WMT_EXTREME_DECISION)
wmt_ocf = ttm_net_income_as_of(ocf_for_quarter_audit, 'WMT', WMT_EXTREME_DECISION)
wmt_extreme_ratio = wmt_ocf.value / wmt_ni.value

wmt_annual_accession = wmt_ocf.quarters.iloc[-1].source_accessions[-1]
wmt_non_cash = payloads['WMT']['facts']['us-gaap'].get('DepreciationAmortizationAndAccretionNet', {}).get('units', {}).get('USD', [])
wmt_non_cash_same_filing = [x for x in wmt_non_cash if x.get('accn') == wmt_annual_accession and x.get('end') == '2024-01-31']
pd.DataFrame([{'issuer': 'WMT', 'decision_time': WMT_EXTREME_DECISION, 'raw_OCF_to_NI': wmt_extreme_ratio, 'OCF_TTM_PIT': wmt_ocf.value, 'NI_TTM_PIT': wmt_ni.value, 'annual_accession': wmt_annual_accession, 'same_filing_D_and_A': [x.get('val') for x in wmt_non_cash_same_filing]}])


For WMT's high observed ratio (about 2.30), the same annual filing contains a reported depreciation/amortization fact of $11.853bn alongside OCF $35.726bn and Net Income $15.511bn. That supports the narrow observation that non-cash reconciliation items exist, but this companyfacts extract alone does not establish a complete causal decomposition of the ratio; no working-capital or one-off explanation is asserted. For JPM's negative OCF/positive NI case, the same raw tag is present but a bank's cash-flow statement and financial-instrument activity have distinct economics. The audit therefore records the sign behavior without claiming causality from the aggregate facts.


## 9. Audit verdicts — Cash Realization remains pending

1. **OCF_TTM_PIT reconstruction: CONDITIONAL GO.** The common SEC tag, duration structure, PIT provenance, and compatible cumulative identities support deterministic reconstruction when all required facts/vintages are valid. Duplicate contexts and amendments require the same prospective-vintage policy used in the audit; unavailable or incompatible operands must remain `NA`.

2. **Raw `OCF_TTM_PIT / NI_TTM_PIT` as final Cash Realization representation: NO-GO / REQUIRES TRANSFORMATION.** The fixed conceptual rule is `NA` for `NI_TTM_PIT <= 0`. Even for positive NI, the quotient can be negative, unstable near `NI = 0+`, and materially above one. Values above one signify more cash than reported earnings over the interval, but do not automatically mean proportionally more Quality. A bounded or alternative representation may reduce instability but would discard magnitude/sign information; that trade-off is a separate research decision.

Cash Realization is **not frozen or implemented** here. Cash Realization, Balance-sheet Resilience, the Quality composite, and `X_char` remain unconstructed.


---

# Stage 1C frozen implementation — Cash Realization

The audit decision is now frozen. For valid PIT inputs:

`CashRealization = min(OCF_TTM_PIT / NI_TTM_PIT, 1.0)` only when `NI_TTM_PIT > 0`; otherwise it is `NA`. The cap is upper-only: it discards excess OCF above earnings but never floors negative OCF.


In [ ]:
from pit_cash_realization import cash_realization_as_of

# Preserve the audited raw tag; this adapter merely gives the implementation's required input name.
ocf_implementation_facts = ocf_raw.rename(columns={'taxonomy_tag': 'tag'})

cash_realization_examples = {
    'AAPL_ratio_below_one': cash_realization_as_of(net_income, ocf_implementation_facts, 'AAPL', '2025-11-03T14:30:00Z'),
    'XOM_upper_cap': cash_realization_as_of(net_income, ocf_implementation_facts, 'XOM', '2026-02-19T14:30:00Z'),
    'JPM_negative_ocf': cash_realization_as_of(net_income, ocf_implementation_facts, 'JPM', '2026-02-17T14:30:00Z'),
    'XOM_nonpositive_ni': cash_realization_as_of(net_income, ocf_implementation_facts, 'XOM', '2021-02-25T14:30:00Z'),
}

aapl_cr = cash_realization_examples['AAPL_ratio_below_one']
xom_cr = cash_realization_examples['XOM_upper_cap']
jpm_cr = cash_realization_examples['JPM_negative_ocf']
xom_nonpositive_ni = cash_realization_examples['XOM_nonpositive_ni']
np.testing.assert_allclose(aapl_cr.raw_ratio, 0.995286135166503)
np.testing.assert_allclose(aapl_cr.cash_realization, aapl_cr.raw_ratio)
assert xom_cr.raw_ratio > 1 and xom_cr.cash_realization == 1.0
assert jpm_cr.raw_ratio < 0 and jpm_cr.cash_realization == jpm_cr.raw_ratio
assert xom_nonpositive_ni.status == 'NONPOSITIVE_NET_INCOME' and xom_nonpositive_ni.cash_realization is None
np.testing.assert_allclose(aapl_ocf.value, aapl_cr.operating_cash_flow.value)


## 10. Representative frozen-rule validation

The table makes all four economically distinct cases visible. It is an implementation consistency check, not a cross-sectional score or a forecast.


In [ ]:
pd.DataFrame([{
    'case': name,
    'issuer': result.issuer,
    'decision_time': result.decision_time,
    'OCF_TTM_PIT': result.operating_cash_flow.value,
    'NI_TTM_PIT': result.net_income.value,
    'raw_OCF_to_NI': result.raw_ratio,
    'CashRealization': result.cash_realization,
    'status': result.status,
} for name, result in cash_realization_examples.items()])


Interpretation of the shown cases:

- AAPL: ratio about 0.995 remains about 0.995; earnings are nearly fully backed, but not capped.
- XOM: OCF/NI is above one, so Cash Realization is exactly one; excess OCF is intentionally not interpreted as proportionally more Quality.
- JPM: OCF is negative while NI is positive, so Cash Realization stays negative; no lower floor is applied.
- XOM 2021: valid TTM Net Income is negative, so Cash Realization is `NA` even though an OCF TTM exists.

Cash Realization is now frozen as a standalone raw Quality component. Balance-sheet Resilience, the Quality composite, cross-sectional transformations, and `X_char` remain unimplemented.
